# 3 · Prototype case studies

The prototype (`prototype/app.py`) takes a location and a start month, downloads every raw Sentinel-1 scene of 12
months, predicts a map per month with E1 (month mean), and measures **built-up change** with a robust rule (built in
most of the last 3 months and in none of the first 3; patches under 400 m² dropped). Each place is checked against two
independent products never used in training: **Open Buildings 2.5D Temporal** (yearly building presence; Africa,
South / Southeast Asia, Latin America) and **ESRI 10 m land cover**. The verdict compares the new built-up with the
model's measured error level on unseen test places.

The examples were computed with `prototype/case_studies.py` (July 2022 – June 2023, 5.12 km × 5.12 km) and include
places chosen *because* they are hard for radar.

In [ ]:
import json, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent                  # the s1-lulc folder
sys.path[:0] = [str(ROOT / "data"), str(ROOT / "train"), str(ROOT / "prototype")]
DATA, PREP, RUNS = ROOT / "data" / "tiles_v0", ROOT / "data" / "tiles_v0" / "prepared", ROOT / "train" / "runs"
from classes import CLASS_NAMES, DISPLAY_NAMES, CLASS_COLOURS, N_CLASSES, BUILT
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project folder:", ROOT)

In [ ]:
import gzip, pickle
import pipeline as P
CASES = sorted((ROOT / "prototype" / "case_studies").glob("*.pkl.gz"))
cases = []
for f in CASES:
    with gzip.open(f, "rb") as fh:
        cases.append(pickle.load(fh))
print(len(cases), "places:", ", ".join(c["case"]["title"] for c in cases))

In [ ]:
def built_refs(res, bc):
    refs = []
    if res.get("obt"):
        o = res["obt"]
        refs.append(P.reference_change(f"Open Buildings Temporal ({o['date_a'][:4]} vs {o['date_b'][:4]})",
                                       o["built_a"], o["built_b"], o["valid"], bc))
    if res.get("esri_ab"):
        y0, y1, ea, eb = res["esri_ab"]
        refs.append(P.reference_change(f"ESRI land cover ({y0} vs {y1})", ea == P.BUILT, eb == P.BUILT,
                                       (ea >= 0) & (eb >= 0), bc))
    return refs

summary = []
for res in cases:
    c = res["case"]; bc = P.built_change(res["pred"], res["months"])
    v, why, exp = P.change_verdict(bc, c["floor"])
    refs = built_refs(res, bc)
    row = {"place": c["title"], "type": c["kind"], "scenes": int(res["n"].sum()),
           "built-up start %": bc["before_pct"], "end %": bc["after_pct"], "new ha": bc["gain_ha"],
           "lost ha": bc["loss_ha"], "error level ha": exp, "verdict": v}
    for r in refs:
        key = "OBT" if r["source"].startswith("Open") else "ESRI"
        row[f"{key} new ha"] = r["new built-up (ha)"]; row[f"{key} confirms %"] = r["our new built-up that it confirms (%)"]
    summary.append(row)
display(pd.DataFrame(summary).set_index("place").round(1))

## Each place in detail

In [ ]:
from IPython.display import display
for res in cases:
    c = res["case"]; months = res["months"]; bc = P.built_change(res["pred"], months)
    v, why, _ = P.change_verdict(bc, c["floor"])
    display(Markdown(f"### {c['title']}  ·  *{c['kind']}*\n\n**What to look for:** {c['note']}\n\n**Verdict: {v}.** {why}"))
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={"width_ratios": [1, 1, 1.4]})
    ax[0].imshow(P.colourise(P.year_majority(res["pred"]))); ax[0].set_title("year majority (our model)")
    ax[1].imshow(P.change_rgb(bc)); ax[1].set_title("built-up change: magenta new, cyan lost")
    pct = P.class_percentages(res["pred"], months)
    for k, col in enumerate(pct.columns):
        ax[2].plot(range(12), pct[col], color=CLASS_COLOURS[k], label=col, marker="o", ms=3)
    ax[2].set_xticks(range(12), [m[2:] for m in months], rotation=45, fontsize=8); ax[2].set_ylabel("% of area")
    ax[2].legend(fontsize=7, ncol=2); ax[2].set_title("class % per month")
    for a in ax[:2]: a.axis("off")
    plt.tight_layout(); plt.show()
    display(P.built_change_table(bc, built_refs(res, bc)).round(2))
    pc = P.period_change(res["pred"], months)
    if pc is not None:
        display(pc.round(1))